In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tqdm import tqdm
import sys


sys.path.insert(0,'/content/drive/MyDrive/Wunderfund/')
import utils
import myfunc as mf

In [ ]:
def get_vecs(dataset, start_index, end_index, history_size, target_size):
  data = []
  labels = []

  start_index = start_index + history_size
  if end_index is None:
    end_index = len(dataset) - target_size

  for i in range(start_index, end_index):
    indices = range(i-history_size, i)
    # Reshape data from (history_size,) to (history_size, 1)
    # data.append(np.reshape(dataset[indices], (history_size, 1)))
    data.append(dataset.iloc[indices])
    labels.append(dataset.iloc[i+target_size])
  return np.array(data), np.array(labels)

In [ ]:
df = pd.read_parquet("/content/drive/MyDrive/Wunderfund/datasets/train.parquet")

In [ ]:
ft = [str(i) for i in range(32)]
df[ft] = df[ft].astype('float32')

In [ ]:
beg = 400
hist = 101
tr_bond = 470
end = 517

In [ ]:
a = df.query("seq_ix == 437")
a.to_csv("/content/drive/MyDrive/Wunderfund/datasets/s437.csv", columns=a.columns)

In [ ]:
a = df.query(f"seq_ix == {beg}").iloc[:,3:]
X, Y = get_vecs(a, 0, None, hist, 0)
for seq_ix in tqdm(range(beg+1, tr_bond)):
  a = df.query(f"seq_ix == {seq_ix}").iloc[:,3:]
  x, y = get_vecs(a, 0, None, hist, 0)

  X = np.concatenate((X, x))
  Y = np.concatenate((Y, y))

In [ ]:
a = df.query(f"seq_ix == {tr_bond}").iloc[:,3:]
U, V = get_vecs(a, 0, None, hist, 0)
for seq_ix in tqdm(range(tr_bond+1, end)):
  a = df.query(f"seq_ix == {seq_ix}").iloc[:,3:]
  x, y = get_vecs(a, 0, None, hist, 0)

  U = np.concatenate((U, x))
  V = np.concatenate((V, y))

In [ ]:
X.shape, Y.shape

In [ ]:
BATCH_SIZE = 256
BUFFER_SIZE = 10000

tr = tf.data.Dataset.from_tensor_slices((X, Y))
tr = tr.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE).repeat()

val = tf.data.Dataset.from_tensor_slices((U, V))
val_ = val.batch(BATCH_SIZE).repeat()

In [ ]:
model= tf.keras.models.Sequential([
    tf.keras.layers.LSTM(128, input_shape=X.shape[-2:]),
    tf.keras.layers.Dense(32)
])
model.compile(optimizer='adam', loss='mse')

In [ ]:
model= tf.keras.models.Sequential([
    tf.keras.layers.GRU(64, input_shape=X.shape[-2:], return_sequences=True),
    tf.keras.layers.GRU(32),
    tf.keras.layers.Dense(32),
])
model.compile(optimizer='adam', loss='mse')

In [ ]:
for x, y in val_.take(1):
    print(model.predict(x).shape)

In [ ]:
EVALUATION_INTERVAL = 500
EPOCHS = 5

model.fit(tr, epochs=EPOCHS,
          steps_per_epoch=EVALUATION_INTERVAL,
          validation_data=val_, validation_steps=50)

In [ ]:
rg = [*range(250, 260)]
x_, y_true = get_vecs(df.query(f"seq_ix in {rg}").iloc[:,3:], 0, None, hist, 0)

In [ ]:
y_pred = model.predict(x_)

In [ ]:
x_.shape

In [ ]:
metric = tf.keras.metrics.R2Score()
metric.update_state(y_true, y_pred)
result = metric.result()
print(result)

In [ ]:
model.save("/content/drive/MyDrive/Wunderfund/model2810_2.keras")

In [ ]:
from utils import DataPoint, ScorerStepByStep

# no need get_vecs here

class PredictionModel:

    def __init__(self):
        self.current_seq_ix = None
        self.sequence_history = []
        self.model = tf.keras.models.load_model('/content/drive/MyDrive/Wunderfund/model1610.keras')

    def predict(self, data_point: DataPoint) -> np.ndarray:
        if self.current_seq_ix != data_point.seq_ix:
            self.current_seq_ix = data_point.seq_ix
            self.sequence_history = []

        self.sequence_history.append(data_point.state.copy().reshape(1, 32))
        if len(self.sequence_history) > 101:
          self.sequence_history = self.sequence_history[1:]

        if not data_point.need_prediction:
            return None

        x = np.concatenate(self.sequence_history)
        global DEBUG
        DEBUG = x
        y_pred = self.model.predict(x.reshape(1, 101, 32))

        return y_pred

In [ ]:
test = df.query("seq_ix == 270")
test.iloc[0].tolist()
test

In [ ]:
from time import sleep
md = PredictionModel()

for i in range(100):
  line = test.iloc[i].tolist()
  dp = DataPoint(int(line[0]), int(line[1]), bool(line[2]), np.array(line[3:]))
  y_pred = md.predict(dp)
  assert y_pred is None


y_pred = []
y_true = []
for i in range(100, 300):
  line = test.iloc[i].tolist()
  dp = DataPoint(int(line[0]), int(line[1]), bool(line[2]), np.array(line[3:]))
  if i < 299:
    y_pred.append(md.predict(dp))
  if i > 100:
    y_true.append(np.array(line[3:]).reshape(1, 32))


In [ ]:
line = test.iloc[100].tolist()
dp = DataPoint(int(line[0]), int(line[1]), bool(line[2]), np.array(line[3:]))
res = md.predict(dp)

In [ ]:
res

In [ ]:
YP = np.concatenate(y_pred)
YT = np.concatenate(y_true)

In [ ]:
u, v = get_vecs(test.iloc[:,3:], 0, 300, 101, 0)

# Конвертация модели

In [ ]:
!pip install tf2onnx==1.16.1
!pip install numpy==1.26.4
!pip install onnxruntime

In [ ]:
import tf2onnx

model = tf.keras.models.load_model('/content/drive/MyDrive/Wunderfund/model1610.keras')

if isinstance(model, tf.keras.Sequential):
    input_shape = model.input_shape
    if isinstance(input_shape, list):
        input_shape = input_shape[0]

    input_layer = tf.keras.layers.Input(shape=input_shape[1:], name='input')
    x = input_layer
    for layer in model.layers:
        x = layer(x)
    functional_model = tf.keras.Model(inputs=input_layer, outputs=x)
else:
    functional_model = model

input_signature = [tf.TensorSpec(functional_model.inputs[0].shape, tf.float32, name='input')]

# Конвертация в ONNX
try:
    onnx_model, _ = tf2onnx.convert.from_keras(functional_model, input_signature, opset=13)
except Exception as e:
    print(f"Ошибка при конвертации: {e}")
    onnx_model = tf2onnx.convert.from_keras(functional_model, opset=13).model

# Сохранение ONNX-модели
with open('model.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())

print("Модель успешно конвертирована в ONNX!")

In [ ]:
arr = np.random.uniform(low=-4, high=4, size=(1, 101, 32))
arr.astype(np.float32)

In [ ]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession('model.onnx')

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
dummy_input = arr.astype(np.float32)

result = session.run([output_name], {input_name: dummy_input})
print(result[0])

In [ ]:
yo = result[0]
yt = model.predict(arr.astype(np.float32))
np.linalg.norm(yo[0] - yt[0])

# Исследование исходных данных

In [ ]:
df.describe().T

In [ ]:
df.query("seq_ix == 37")['17'].plot.hist(bins=30, alpha=0.7, edgecolor='black')
plt.title('Распределение значений')
plt.xlabel('Значения')
plt.ylabel('Частота')
plt.show()